# Extracción de texto con base en el contexto

## Librerías

In [11]:
%%capture
!pip install bs4 lxml html2text python-dotenv dgml-utils

In [10]:
import pathlib
import re
import os
from functools import partial
from typing import Generator

from bs4 import BeautifulSoup, Doctype, NavigableString, SoupStrainer, Tag
from dotenv import load_dotenv
from html2text import HTML2Text
from IPython.core.display import Markdown
from langchain_community.document_loaders import DocugamiLoader, RecursiveUrlLoader

load_dotenv() # Debería devolver True

True

## Web

### Dataset y función de utilidad

Para mas informacion acerca de partial puedes visitar: [https://docs.python.org/3/library/functools.html#functools.partial](https://docs.python.org/3/library/functools.html#functools.partial)

In [4]:
doc_url = "https://docs.langchain.com/oss/python/langchain/install"

load_documents = partial(
    RecursiveUrlLoader,
    url=doc_url,
    max_depth=3,
    prevent_outside=True,
    check_response_status=True,
)

### Extracción de texto sin tener en cuenta el contexto

La primera aproximación para extraer texto de una página web es simplemente obtener el texto de todos los elementos de la página.

In [5]:
def webpage_text_extractor(html: str) -> str:
    return BeautifulSoup(html, "lxml").get_text(separator="\n", strip=True)


loader = load_documents(
    extractor=webpage_text_extractor,
)

docs_without_data_context = loader.load()
print(docs_without_data_context[0].page_content[:520])

Unable to load from https://docs.langchain.com/cdn-cgi/l/email-protection. Received error Received HTTP status 404 of type ValueError


Install LangChain - Docs by LangChain
Skip to main content
Docs by LangChain
home page
LangChain + LangGraph
Search...
⌘
K
Support
GitHub
Try LangSmith
Try LangSmith
Search...
Navigation
Get started
Install LangChain
LangChain
LangGraph
Deep Agents
Integrations
Learn
Reference
Contribute
Python
Overview
Get started
Install
Quickstart
Changelog
Philosophy
Core components
Agents
Models
Messages
Tools
Short-term memory
Streaming
Structured output
Middleware
Overview
Built-in middleware
Custom middleware
Advanced usage


### Extracción de texto teniendo un poco de contexto

El texto de la documentación de `Langchain` está escrito en `Markdown`, teniendo una estructura que puede ser aprovechada para extraer el texto de manera más precisa. Para ello, utilizaremos una librería que nos permita convertir el texto de `HTML` a `Markdown`.

In [6]:
def markdown_extractor(html: str) -> str:
    html2text = HTML2Text()
    html2text.ignore_links = False
    html2text.ignore_images = False
    return html2text.handle(html)


loader = load_documents(
    extractor=markdown_extractor,
)

docs_with_a_bit_of_context = loader.load()
print(docs_with_a_bit_of_context[0].page_content[:3000])

Unable to load from https://docs.langchain.com/cdn-cgi/l/email-protection. Received error Received HTTP status 404 of type ValueError


Skip to main content

[Docs by LangChain home page![light
logo](https://mintcdn.com/langchain-5e9cc07a/Xbr8HuVd9jPi6qTU/images/brand/langchain-
docs-
teal.svg?fit=max&auto=format&n=Xbr8HuVd9jPi6qTU&q=85&s=16111530672bf976cb54ef2143478342)![dark
logo](https://mintcdn.com/langchain-5e9cc07a/Xbr8HuVd9jPi6qTU/images/brand/langchain-
docs-
lilac.svg?fit=max&auto=format&n=Xbr8HuVd9jPi6qTU&q=85&s=b70fb1a2208670492ef94aef14b680be)](/)

LangChain + LangGraph

Search...

⌘K

  * [Support](https://support.langchain.com/)
  * [GitHub](https://github.com/langchain-ai)
  * [Try LangSmith](https://smith.langchain.com/)
  * [Try LangSmith](https://smith.langchain.com/)

Search...

Navigation

Get started

Install LangChain

[LangChain](/oss/python/langchain/overview)[LangGraph](/oss/python/langgraph/overview)[Deep
Agents](/oss/python/deepagents/overview)[Integrations](/oss/python/integrations/providers/overview)[Learn](/oss/python/learn)[Reference](/oss/python/reference/overview)[Contribute](/oss/pyth

### Extracción de texto teniendo en cuenta el contexto

Si bien, cuando utilizamos una librería para convertir el texto de `HTML` a `Markdown` pudimos extraer el texto de manera más precisa, aún hay algunos casos en los que no se logra extraer el texto de manera correcta.

Es aquí donde entra en juego el dominio del problema. Con base en el conocimiento que tenemos del problema, podemos crear una función que nos permita extraer el texto de manera más precisa.

Imagina que `langchain_docs_extractor` es como un obrero especializado en una fábrica cuyo trabajo es transformar materias primas (documentos HTML) en un producto terminado (un string limpio y formateado). Este obrero usa una herramienta especial, `get_text`, como una máquina para procesar las materias primas en piezas utilizables, examinando cada componente de la materia prima **pieza por pieza**, y usa el mismo proceso repetidamente (**recursividad**) para descomponer los componentes en su forma más simple. Al final, ensambla todas las piezas procesadas en un producto completo y hace algunos refinamientos finales antes de que el producto salga de la fábrica.

In [7]:
def langchain_docs_extractor(
    html: str,
    include_output_cells: bool,
    path_url: str | None = None,
) -> str:
    # 1. Strainer optimizado para el ID del contenedor
    soup = BeautifulSoup(
        html,
        "lxml",
        parse_only=SoupStrainer(id="content-container"),
    )

    # 2. Eliminación de ruido específico del nuevo formato (Sidebar, TOC, Header)
    # 'content-side-layout' contiene la navegación derecha "On this page"
    SCAPE_IDS = ["content-side-layout", "table-of-contents", "header", "menu"]
    for div_id in SCAPE_IDS:
        for tag in soup.find_all(id=div_id):
            tag.decompose()

    # Eliminación de tags generales
    SCAPE_TAGS = ["nav", "footer", "aside", "script", "style", "button", "noscript"]
    [tag.decompose() for tag in soup.find_all(SCAPE_TAGS)]

    def get_text(tag: Tag) -> Generator[str, None, None]:
        for child in tag.children:
            if isinstance(child, Doctype):
                continue

            if isinstance(child, NavigableString):
                yield child.get_text()
            
            elif isinstance(child, Tag):
                # --- Encabezados ---
                if child.name in ["h1", "h2", "h3", "h4", "h5", "h6"]:
                    yield f"\n\n{'#' * int(child.name[1:])} "
                    yield child.get_text(strip=True)
                    
                    if path_url is not None:
                        link = child.find("a")
                        if link is not None and link.has_attr('href'):
                            yield f" [](/{path_url}/{link.get('href')})"
                    yield "\n\n"

                # --- Enlaces e Imágenes ---
                elif child.name == "a":
                    href = child.get('href', '')
                    text = child.get_text(strip=True)
                    if text and href:
                        yield f"[{text}]({href})"
                    else:
                        yield text
                elif child.name == "img":
                    src = child.get('src', '')
                    alt = child.get('alt', '')
                    yield f"![{alt}]({src})"

                # --- Formato de texto ---
                elif child.name in ["strong", "b"]:
                    yield f"**{child.get_text(strip=True)}**"
                elif child.name in ["em", "i"]:
                    yield f"_{child.get_text(strip=True)}_"
                elif child.name == "br":
                    yield "\n"

                # --- Bloques de Código ---
                elif child.name == "code":
                    parent = child.find_parent()
                    # Verificamos si es un bloque de código multilinea (dentro de un pre)
                    if parent is not None and parent.name == "pre":
                        # Intentar detectar lenguaje (Mintlify suele ponerlo en el div padre o clases)
                        language = ""
                        grandparent = parent.find_parent("div")
                        
                        # Estrategia 1: Atributo language en el padre
                        if grandparent and grandparent.has_attr("language"):
                            language = grandparent.get("language")
                        # Estrategia 2: Clases antiguas
                        elif parent.has_attr("class"):
                            classes = parent.attrs.get("class", [])
                            lang_match = next(filter(lambda x: re.match(r"language-\w+", x), classes), None)
                            if lang_match:
                                language = lang_match.split("-")[1]

                        # Filtrado opcional de celdas de salida
                        if language in ["pycon", "text"] and not include_output_cells:
                            continue

                        code_content = child.get_text()
                        yield f"\n```{language}\n{code_content}\n```\n\n"
                    else:
                        # Código inline
                        yield f"`{child.get_text(strip=False)}`"

                # --- Párrafos (Soporte para <p> y <span data-as="p">) ---
                elif child.name == "p" or (child.name == "span" and child.get("data-as") == "p"):
                    yield from get_text(child)
                    yield "\n\n"

                # --- Listas ---
                elif child.name == "ul":
                    # Chequeo simple si es referencia de API (lógica antigua conservada por si acaso)
                    is_api = "api_reference" in child.attrs
                    prefix = "> - " if is_api else "- "
                    
                    for li in child.find_all("li", recursive=False):
                        yield prefix
                        yield from get_text(li)
                        yield "\n"
                    yield "\n\n"
                
                elif child.name == "ol":
                    for i, li in enumerate(child.find_all("li", recursive=False)):
                        yield f"{i + 1}. "
                        yield from get_text(li)
                        yield "\n\n"

                # --- Tablas ---
                elif child.name == "table":
                    thead = child.find("thead")
                    if isinstance(thead, Tag):
                        headers = thead.find_all("th")
                        if headers:
                            yield "| " + " | ".join(h.get_text(strip=True) for h in headers) + " |\n"
                            yield "| " + " | ".join("---" for _ in headers) + " |\n"

                    tbody = child.find("tbody")
                    if isinstance(tbody, Tag):
                        for row in tbody.find_all("tr"):
                            cells = row.find_all(["td", "th"])
                            yield "| " + " | ".join(c.get_text(strip=True) for c in cells) + " |\n"
                    yield "\n\n"

                # --- Recursividad por defecto ---
                else:
                    yield from get_text(child)

    joined = "".join(get_text(soup))
    return re.sub(r"\n\n+", "\n\n", joined).strip()


loader = load_documents(
    extractor=partial(
        langchain_docs_extractor,
        include_output_cells=True,
    ),
)

docs_with_data_context = loader.load()
print(docs_with_data_context[0].page_content)

Unable to load from https://docs.langchain.com/cdn-cgi/l/email-protection. Received error Received HTTP status 404 of type ValueError


To install the LangChain package:

Copy
```
pip install -U langchain
# Requires Python 3.10+

```

LangChain provides integrations to hundreds of LLMs and thousands of other integrations. These live in independent provider packages.

Copy
```
# Installing the OpenAI integration
pip install -U langchain-openai

# Installing the Anthropic integration
pip install -U langchain-anthropic

```

See the [Integrations tab](/oss/python/integrations/providers/overview) for a full list of available integrations.

Now that you have LangChain installed, you can get started by following the [Quickstart guide](/oss/python/langchain/quickstart).

[Edit this page on GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/install.mdx) or [file an issue](https://github.com/langchain-ai/docs/issues/new/choose).

[Connect these docs](/use-these-docs) to Claude, VSCode, and more via MCP for real-time answers.

Was this page helpful?

[LangChain overviewPrevious](/oss/python/langchain/overvi

El archivo de salida es ahora en formato Markdown, lo que permite visualizarlo en cualquier editor de texto o en GitHub, ofreciendo una estructura de la información más clara y accesible. Esta organización permite realizar cortes de texto con mayor precisión, facilitando así la obtención de información más pertinente y relevante.

In [8]:
Markdown(docs_with_data_context[0].page_content)

To install the LangChain package:

Copy
```
pip install -U langchain
# Requires Python 3.10+

```

LangChain provides integrations to hundreds of LLMs and thousands of other integrations. These live in independent provider packages.

Copy
```
# Installing the OpenAI integration
pip install -U langchain-openai

# Installing the Anthropic integration
pip install -U langchain-anthropic

```

See the [Integrations tab](/oss/python/integrations/providers/overview) for a full list of available integrations.

Now that you have LangChain installed, you can get started by following the [Quickstart guide](/oss/python/langchain/quickstart).

[Edit this page on GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/install.mdx) or [file an issue](https://github.com/langchain-ai/docs/issues/new/choose).

[Connect these docs](/use-these-docs) to Claude, VSCode, and more via MCP for real-time answers.

Was this page helpful?

[LangChain overviewPrevious](/oss/python/langchain/overview)[QuickstartNext](/oss/python/langchain/quickstart)⌘I

## PDF / DOCX / DOC

### Dataset de prueba

En este ejemplo, vamos a emplear algunos archivos de muestra proporcionados por [Docugami](https://www.docugami.com/). Dichos archivos representan el producto de la extracción de texto de documentos auténticos, en particular, de archivos PDF relativos a contratos de arrendamiento comercial.

In [9]:
lease_data_dir = pathlib.Path("./data/docugami/commercial_lease")
lease_files = list(lease_data_dir.glob("*.xml"))
lease_files

[PosixPath('data/docugami/commercial_lease/TruTone Lane 6.xml'),
 PosixPath('data/docugami/commercial_lease/TruTone Lane 5.xml'),
 PosixPath('data/docugami/commercial_lease/TruTone Lane 4.xml'),
 PosixPath('data/docugami/commercial_lease/TruTone Lane 1.xml'),
 PosixPath('data/docugami/commercial_lease/TruTone Lane 3.xml'),
 PosixPath('data/docugami/commercial_lease/TruTone Lane 2.xml')]

Ahora, carguemos los documentos de muestra y veamos qué propiedades tienen.

In [12]:
loader = DocugamiLoader(
    docset_id=None,
    access_token=None,
    document_ids=None,
    file_paths=lease_files,
)

lease_docs = loader.load()
f"Loaded {len(lease_docs)} documents."

'Loaded 1070 documents.'

La metadata obtenida del documento incluye los siguientes elementos:

- `id`, `source_id` y `name`: Estos campos identifican de manera unívoca al documento y al fragmento de texto que se ha extraído de él.
- `xpath`: Es el `XPath` correspondiente dentro de la representación XML del documento. Se refiere específicamente al fragmento extraído. Este campo es útil para referenciar directamente las citas del fragmento real dentro del documento XML.
- `structure`: Incluye los atributos estructurales del fragmento, tales como `p`, `h1`, `div`, `table`, `td`, entre otros. Es útil para filtrar ciertos tipos de fragmentos, en caso de que el usuario los requiera.
- `tag`: Representa la etiqueta semántica para el fragmento. Se genera utilizando diversas técnicas, tanto generativas como extractivas, para determinar el significado del fragmento en cuestión.

In [13]:
lease_docs[0].metadata

{'xpath': '/docset:OFFICELEASEAGREEMENT-section/dg:chunk',
 'id': 'f590fff603222021a19558237451df87',
 'name': 'TruTone Lane 6.xml',
 'source': 'TruTone Lane 6.xml',
 'structure': 'h1 h1',
 'tag': 'chunk'}

`Docugami` también posee la capacidad de asistir en la extracción de metadatos específicos para cada `chunk` o fragmento de nuestros documentos. A continuación, se presenta un ejemplo de cómo se extraen y representan estos metadatos:

```json
{
    'xpath': '/docset:OFFICELEASEAGREEMENT-section/docset:OFFICELEASEAGREEMENT/docset:LeaseParties',
    'id': 'v1bvgaozfkak',
    'source': 'TruTone Lane 2.docx',
    'structure': 'p',
    'tag': 'LeaseParties',
    'Lease Date': 'April 24 \n\n ,',
    'Landlord': 'BUBBA CENTER PARTNERSHIP',
    'Tenant': 'Truetone Lane LLC',
    'Lease Parties': 'Este ACUERDO DE ARRENDAMIENTO DE OFICINA (el "Contrato") es celebrado por y entre BUBBA CENTER PARTNERSHIP ("Arrendador"), y Truetone Lane LLC, una compañía de responsabilidad limitada de Delaware ("Arrendatario").'
}
```

Los metadatos adicionales, como los mostrados arriba, pueden ser extremadamente útiles cuando se implementan `self-retrievers`, los cuales serán explorados a detalle más adelante.

### Carga tus documentos

Si prefieres utilizar tus propios documentos, puedes cargarlos a través de la interfaz gráfica de [Docugami](https://www.docugami.com/). Una vez cargados, necesitarás asignar cada uno a un `docset`. Un `docset` es un conjunto de documentos que presentan una estructura análoga (necesitarás al menos 6 documentos). Por ejemplo, todos los contratos de arrendamiento comercial por lo general poseen estructuras similares, por lo que pueden ser agrupados en un único `docset`.

Después de crear tu `docset`, los documentos cargados serán procesados y estarán disponibles para su acceso mediante la API de `Docugami`.

Para recuperar los `ids` de tus documentos y de sus correspondientes `docsets`, puedes ejecutar el siguiente comando:

```bash
curl --header "Authorization: Bearer {YOUR_DOCUGAMI_TOKEN}" \
  https://api.docugami.com/v1preview1/documents
```

Este comando te facilitará el acceso a la información relevante, optimizando así la administración y organización de tus documentos dentro de `Docugami`.

Una vez hayas extraído los `ids` de tus documentos o de los `docsets`, podrás emplearlos para acceder a la información de tus documentos utilizando el `DocugamiLoader` de `Langchain`. Esto te permitirá manipular y gestionar tus documentos dentro de tu aplicación.

In [ ]:
loader = DocugamiLoader(
    docset_id="rqgjwt6chko2",
    access_token=os.getenv('DOCUGAMI_API_KEY'),
    document_ids=None,
    file_paths=None,
)

papers_docs = loader.load()

In [ ]:
papers_docs

In [14]:
lost_in_the_middle_paper_docs = [
    doc for doc in papers_docs if "Developer.pdf" in doc.metadata["source"]
]
for doc in lost_in_the_middle_paper_docs:
    print(doc.metadata["tag"])

chunk DevSecOpsIntegrationSteps
chunk
chunk
chunk
chunk
chunk
chunk
chunk
chunk
SetSupplyChainSecurityinDevSecOps chunk
chunk
chunk
chunk
chunk
DevSecOpsIntroduction
DevSecOpsImplementationChallenges
DevSecOpsIntroductionGuide
table
chunk SoftwareSecurityConcerns
SecurityChallengesinSoftwareDevelopment
table
chunk
SoftwareSupplyChainVulnerability
SupplyChainSecurityRisks
chunk
SupplyChainSecurityImportance
table
table
SupplyChainSecurityInitiative
table
DigitalTransformationChallenges
CybersecurityThreatLandscape
SecurityChallengesinDevOps
chunk
DeveloperSecurityChallenges
CloudSecurityComplexity
table
chunk
TheGrowingUseofOpenSourceComponents
chunk
NewAttackVectorsDiscoveredEachDay
DependencyManagementComplexity
chunk
StricterRegulatoryRequirements.34
CybersecurityandSoftwareSupplyChain
chunk
table
SoftwareSupplyChainSecurityBenefits
chunk SoftwareSupplyChainTrends
table
DevSecOpsStrategyImplementation
EnterpriseChallengesinDeployment
table
chunk CognitiveLoadinDevelopment
chunk Secur